In [1]:
import pandas as pd
import asyncio
import re
from pathlib import Path
from playwright.async_api import async_playwright

# Membaca 50 kandidat
df_50 = pd.read_csv("data/kandidat_50_putusan.csv")

# Folder PDF
FOLDER_PDF = Path("PDF_Putusan")

for tahun in [2023, 2024, 2025]:
    (FOLDER_PDF / str(tahun)).mkdir(
        parents=True,
        exist_ok=True
    )

print("Total kandidat:", len(df_50))
print("Folder PDF siap digunakan.")

Total kandidat: 50
Folder PDF siap digunakan.


In [2]:
def nama_file_aman(no_putusan):
    nama = re.sub(r"[^a-zA-Z0-9]+", "_", str(no_putusan))
    return f"putusan_{nama}.pdf"


async def download_putusan(browser, row):

    tahun = int(row["Tahun Filter"])
    no_putusan = str(row["No Putusan"])
    url_detail = row["URL Detail"]

    folder_tahun = FOLDER_PDF / str(tahun)
    folder_tahun.mkdir(parents=True, exist_ok=True)

    lokasi_file = folder_tahun / nama_file_aman(no_putusan)

    if lokasi_file.exists():
        with open(lokasi_file, "rb") as f:
            if f.read(5) == b"%PDF-":
                return {
                    "No Putusan": no_putusan,
                    "Tahun": tahun,
                    "Status": "BERHASIL_SUDAH_ADA",
                    "File": str(lokasi_file)
                }

    context = browser.contexts[0]
    page = await context.new_page()

    try:
        response = await page.goto(
            url_detail,
            wait_until="domcontentloaded",
            timeout=60000
        )

        if response is None or response.status != 200:
            return {
                "No Putusan": no_putusan,
                "Tahun": tahun,
                "Status": "GAGAL_AKSES",
                "File": None
            }

        await page.wait_for_timeout(2000)

        pdf_links = await page.locator(
            'a[href*="/pdf/"]'
        ).evaluate_all("""
            elements => elements.map(a => a.href)
        """)

        pdf_links = [
            url for url in pdf_links
            if "/zip/" not in url.lower()
        ]

        if not pdf_links:
            return {
                "No Putusan": no_putusan,
                "Tahun": tahun,
                "Status": "PDF_TIDAK_DITEMUKAN",
                "File": None
            }

        pdf_url = pdf_links[0]

        async with page.expect_download(timeout=60000) as info:
            await page.evaluate("""
                (url) => {
                    const link = [...document.querySelectorAll('a')]
                        .find(a => a.href === url);
                    if (link) link.click();
                }
            """, pdf_url)

        download = await info.value
        await download.save_as(str(lokasi_file))

        with open(lokasi_file, "rb") as f:
            header = f.read(5)

        ukuran = lokasi_file.stat().st_size

        if header == b"%PDF-" and ukuran > 0:
            status = "BERHASIL"
        else:
            lokasi_file.unlink(missing_ok=True)
            status = "PDF_TIDAK_VALID"

        return {
            "No Putusan": no_putusan,
            "Tahun": tahun,
            "Status": status,
            "File": str(lokasi_file) if status == "BERHASIL" else None
        }

    except Exception as e:
        lokasi_file.unlink(missing_ok=True)

        return {
            "No Putusan": no_putusan,
            "Tahun": tahun,
            "Status": "ERROR: " + str(e)[:150],
            "File": None
        }

    finally:
        await page.close()

In [3]:
async def jalankan_download():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        hasil = []

        for i, row in df_50.head(5).iterrows():

            print(
                f"[{i+1}/5] Download:",
                row["No Putusan"]
            )

            result = await download_putusan(browser, row)
            hasil.append(result)

            print("Status:", result["Status"])

            # Jeda agar tidak membebani server
            await asyncio.sleep(3)

        df_hasil = pd.DataFrame(hasil)

        df_hasil.to_csv(
            "data/hasil_download_pdf.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print("\nREKAP DOWNLOAD:")
        print(df_hasil["Status"].value_counts())

        print(
            "\nPDF valid:",
            len(list(FOLDER_PDF.glob("*/*.pdf")))
        )


await jalankan_download()

[1/5] Download: 2031/Pid.Sus/2023/PN Sby
Status: BERHASIL
[2/5] Download: 1512/Pid.Sus/2023/PN Sby
Status: BERHASIL
[3/5] Download: 2260/Pid.Sus/2023/PN Sby
Status: BERHASIL
[4/5] Download: 2166/Pid.Sus/2023/PN Sby
Status: BERHASIL
[5/5] Download: 270/Pid.Sus/2023/PN Sby
Status: BERHASIL

REKAP DOWNLOAD:
Status
BERHASIL    5
Name: count, dtype: int64

PDF valid: 5


In [4]:
# Membaca kembali 50 kandidat
df_50 = pd.read_csv("data/kandidat_50_putusan.csv")

# Menyiapkan file pencatatan hasil
file_log = Path("data/hasil_download_pdf.csv")

if file_log.exists():
    df_hasil = pd.read_csv(file_log)
else:
    df_hasil = pd.DataFrame(
        columns=["No Putusan", "Tahun", "Status", "File"]
    )

async def download_sisa_putusan():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        hasil_baru = []

        for i, row in df_50.iterrows():

            tahun = int(row["Tahun Filter"])
            no_putusan = str(row["No Putusan"])

            # Periksa apakah PDF sudah ada dan valid
            file_pdf = (
                FOLDER_PDF /
                str(tahun) /
                nama_file_aman(no_putusan)
            )

            if file_pdf.exists():
                with open(file_pdf, "rb") as f:
                    valid = f.read(5) == b"%PDF-"

                if valid:
                    print(f"[{i+1}/50] SUDAH ADA: {no_putusan}")
                    continue

            print(f"\n[{i+1}/50] Download: {no_putusan}")

            result = await download_putusan(browser, row)
            hasil_baru.append(result)

            print("Status:", result["Status"])

            # Simpan progres setiap dokumen
            if hasil_baru:
                df_update = pd.concat(
                    [df_hasil, pd.DataFrame(hasil_baru)],
                    ignore_index=True
                )

                df_update.to_csv(
                    file_log,
                    index=False,
                    encoding="utf-8-sig"
                )

            # Beri jeda antarpermintaan
            await asyncio.sleep(5)

        await browser.close()

        print("\nPROSES DOWNLOAD SELESAI")

        jumlah_valid = 0

        for tahun in [2023, 2024, 2025]:
            folder = FOLDER_PDF / str(tahun)

            for file in folder.glob("*.pdf"):
                with open(file, "rb") as f:
                    if f.read(5) == b"%PDF-":
                        jumlah_valid += 1

        print("TOTAL PDF VALID:", jumlah_valid)

await download_sisa_putusan()

[1/50] SUDAH ADA: 2031/Pid.Sus/2023/PN Sby
[2/50] SUDAH ADA: 1512/Pid.Sus/2023/PN Sby
[3/50] SUDAH ADA: 2260/Pid.Sus/2023/PN Sby
[4/50] SUDAH ADA: 2166/Pid.Sus/2023/PN Sby
[5/50] SUDAH ADA: 270/Pid.Sus/2023/PN Sby

[6/50] Download: 1982/Pid.Sus/2023/PN Sby
Status: BERHASIL

[7/50] Download: 140/Pid.Sus/2023/PN Sby
Status: BERHASIL

[8/50] Download: 2258/Pid.Sus/2023/PN Sby
Status: BERHASIL

[9/50] Download: 2325/Pid.Sus/2023/PN Sby
Status: BERHASIL

[10/50] Download: 2043/Pid.Sus/2023/PN Sby
Status: BERHASIL

[11/50] Download: 2047/Pid.Sus/2023/PN Sby
Status: BERHASIL

[12/50] Download: 1880/Pid.Sus/2023/PN Sby
Status: BERHASIL

[13/50] Download: 1061 K/PDT.SUS-PHI/2026
Status: BERHASIL

[14/50] Download: 2189/Pid.Sus/2023/PN Sby
Status: BERHASIL

[15/50] Download: 1024/Pid.Sus/2023/PN Sby
Status: BERHASIL

[16/50] Download: 2274/Pid.Sus/2023/PN Sby
Status: BERHASIL

[17/50] Download: 2359/Pid.Sus/2023/PN Sby
Status: BERHASIL

[18/50] Download: 2054/Pid.Sus/2024/PN Sby
Status: BERHASIL